# GISLR — Continuous-signing models: per-frame gloss + null + sign-boundary

**What this notebook does.** Pipeline stage (training), TODO §12.3. It trains
models on **continuous multi-sign streams** instead of isolated clips. At
every frame the model outputs a confidence over every gloss plus **null**
(non-signing), and a **sign-boundary** probability ("the sign in progress
has just ended").

**Why.** The §12.2 baselines (`docs/reports/sentence-baselines.md`) showed
that today's isolated models lose most of their accuracy on continuous
signing through *missed* signs. A fixed "confident for N frames" rule
cannot find the end of both short and long signs: 60% correct even with a
perfect reset, vs 76% isolated. So the model is trained (a) per frame, so
confidence is meaningful throughout a sign, and (b) to signal the boundary
itself.

**Runs** (`experiments/recognition/configs/gislr.continuous.json`, one section each):

| run | arch | what it tests |
|---|---|---|
| **C1** | `gru_continuous` | the main model: per-frame gloss+null CE + boundary BCE, all 250 glosses |
| **C2** | `lstm_continuous` | the same with an LSTM body |
| **C3** | `gru_continuous`, `loss: ctc` | alignment-free CTC (blank = null): what real continuous video would need |
| **Copen** | `gru_continuous`, 20 glosses held out | the model TODO §12.4 teaches new signs to (held-out glosses masked in the cosine head) |

Model: `sb.recognize.architectures.ContinuousRNN`, a 2×256 causal GRU/LSTM on
ME-132 xy (the best registry configuration). It has a **cosine** gloss head,
so a new sign can later be added from the mean embedding of a few examples
without retraining. It keeps the standard isolated read-out, so
`sb-evaluate` scores it canonically (§8).

**Training data** (`sb.recognize.continuous.data`). Every epoch is a new
random partition of `train.csv`'s clips into one-signer, random-order streams
of 1–6 signs:
- 0–15 interpolated gap frames between signs (0 = back-to-back);
- 5–30 rest frames at each end, of two kinds:
  - 70% **lowered**: the pose wrists drop to hip height and each hand
    disappears (NaN) once its wrist leaves the frame. This is how MediaPipe
    sees a resting signer in these selfie-framed videos: hips are out of
    frame in 97% of training frames, and hands are never detected below
    y ≈ 0.9;
  - 30% hands-absent with the pose left at signing height, GISLR-Sentences
    v1's rest;
- clips at **native length** (no 128-frame subsampling).

A sign-stratified 5% of `train.csv` becomes fixed validation streams.
Checkpoints are selected on their **segment accuracy**. `test.csv` and
GISLR-Sentences are never used for selection.

**Artifacts**

| path | content |
|---|---|
| `registry/runs/<run_id>/` | one registry run per training (meta.json every epoch, `best.pt`/`last.pt`, `assets/history.json`) |
| `data/cache/gislr/features/clipbank_v1/<key>/train_{data,offsets}.npy` | NaN-preserving `train.csv` clip bank, ~3 GB, content-addressed |
| `data/cache/gislr/continuous/assets/*.png` | composer preview, learning curves |

**Resumable.** Every run auto-resumes from `last.pt` through its pointer file.
A finished run is never reused; re-running a section after it finishes
starts a new run. Each epoch's stream partition is seeded by the epoch
number, so a resumed epoch sees the same data.

**Evaluation on continuous signing** is not here. It is
`gislr.3.streaming.continuous-eval.ipynb`, which uses the same protocol as
the baselines.

## Setup

In [ ]:
# ============================================================
# Imports, config, paths
# ============================================================
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from sb.core.paths import CACHE_DIR
from sb.core.subsets import get_subset
from sb.recognize.continuous import data as CD
from sb.recognize.continuous.train import CONFIG_PATH, load_config, run_dir_for, run_hyp, train_continuous
from sb.recognize.sources import get_source

CONFIG = load_config(CONFIG_PATH)
ASSETS = CACHE_DIR / "gislr" / "continuous" / "assets"
ASSETS.mkdir(parents=True, exist_ok=True)

print(f"config: {CONFIG_PATH.name} · regime {CONFIG['regime']} · {CONFIG['subset']}/{CONFIG['coords']}")
for name, run in CONFIG["runs"].items():
    over = {k: v for k, v in run_hyp(CONFIG, name).items() if CONFIG["shared"].get(k) != v}
    print(f"  {name:<6} {run['arch']:<16} overrides {over or '-'}  held-out glosses {run.get('n_holdout_glosses', 0)}"
          f"  run dir {run_dir_for(name) or '(not started)'}")
print(f"cuda: {torch.cuda.is_available()}")

## 1. Clip bank + composer preview

The first run builds the NaN-preserving `train.csv` clip bank (~1 min, ~3 GB);
after that it is a cache hit. Two example streams are drawn, one per rest
kind, and plotted with their per-frame targets. This is to check the
composer by eye, not a statistic.

In [ ]:
# ============================================================
# Build/load the clip bank, preview two composed streams
# ============================================================
PREVIEW_SEED = 7

ds = get_source(CONFIG["dataset"])
data_dir = ds.resolve_dir()
sign2idx = ds.label_map(data_dir)
idx2sign = {v: k for k, v in sign2idx.items()}
train_split, _ = ds.canonical_split(data_dir, sign2idx)
subset = get_subset(CONFIG["subset"])
bank = CD.ClipBank.build(train_split, "train", subset, CONFIG["coords"], data_dir)
lay = CD.Layout.of(subset.array, CONFIG["coords"])
null_index = len(sign2idx)
print(f"clip bank: {len(bank)} clips, {bank.data.shape[0]:,} frames x {bank.data.shape[1]} features "
      f"({bank.data.nbytes / 1e9:.2f} GB)")

rng = np.random.default_rng(PREVIEW_SEED)
labels = train_split["label"].to_numpy()
pid = train_split["participant_id"].to_numpy()
wrist_col = {int(r): i for i, r in enumerate(subset.array)}[489 + 16] * 2 + 1  # right pose wrist, y
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=False)
for ax, lowered in zip(axes, (1.0, 0.0)):
    who = rng.choice(np.unique(pid))
    clips = rng.choice(np.flatnonzero(pid == who), 4, replace=False)
    s = CD.compose([bank.clip(c) for c in clips], [int(labels[c]) for c in clips], null_index, lay,
                   {**CONFIG["composer"], "p_lowered_rest": lowered}, rng)
    ax.plot(-s["x"][:, wrist_col], label="right pose wrist (-y)")
    hand_present = np.isfinite(s["x"][:, lay.hand_cols[1][0]]) | np.isfinite(s["x"][:, lay.hand_cols[0][0]])
    ax.plot(np.where(hand_present, np.nanmax(-s["x"][:, wrist_col]) + 0.05, np.nan), "g|", label="a hand detected")
    ax.plot(s["b"] * 0.1 + np.nanmin(-s["x"][:, wrist_col]), label="boundary target (scaled)")
    for t in np.flatnonzero(s["y"] == null_index):
        ax.axvspan(t - 0.5, t + 0.5, color="#dddddd", lw=0)
    for (a, b), c in zip(s["segments"], clips):
        ax.text((a + b) / 2, ax.get_ylim()[1], idx2sign[int(labels[c])].upper(), ha="center", va="bottom")
    ax.set_title(f"{'lowered-hands' if lowered else 'hands-absent'} rest · grey = null frames")
    ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(ASSETS / "composer_preview.png", dpi=110)
plt.show()

## 2. Wiring check (seconds; writes nothing)

A few training batches and a short validation pass per run through the
real driver, with no registry folder and no pointer. It catches a broken
config before hours of training.

In [ ]:
# ============================================================
# Smoke every run: a few batches, no files written
# ============================================================
SMOKE_BATCHES = 3

for name in CONFIG["runs"]:
    r = train_continuous(name, smoke=SMOKE_BATCHES)
    print(f"{name:<6} params {r['n_params']:,} · train clips {r['n_train_clips']} "
          f"({r['n_train_sequences']} streams/epoch) · val streams {r['n_val_sequences']} · "
          f"held out {len(r['held_out'])} · first/last loss {r['train_losses'][0][0]:.3f} -> "
          f"{r['train_losses'][-1][0]:.3f}")

## 3. C1: GRU, per-frame + boundary (main model)

The model the continuous evaluation centers on. Auto-resumes if interrupted.

In [ ]:
# ============================================================
# Train C1
# ============================================================
run_dir = train_continuous("C1")

## 4. C2: LSTM body

C1 with `lstm_continuous`; everything else identical. Auto-resumes if interrupted.

In [ ]:
# ============================================================
# Train C2
# ============================================================
run_dir = train_continuous("C2")

## 5. C3: CTC

Trained with CTC (blank = null): no frame alignment used. Its boundary head receives no loss, so decode it with greedy CTC (D4) or null-gated commits (D3). Auto-resumes if interrupted.

In [ ]:
# ============================================================
# Train C3
# ============================================================
run_dir = train_continuous("C3")

## 6. C-open: 20 glosses held out (for §12.4)

C1's recipe with 20 seeded glosses removed from training entirely and masked in the head. Its canonical isolated accuracy on those 20 is 0 by construction until §12.4 enrolls them. Auto-resumes if interrupted.

In [ ]:
# ============================================================
# Train Copen
# ============================================================
run_dir = train_continuous("Copen")

## 7. Learning curves

Read from each run's `assets/history.json`, so this cell works without the
checkpoints and without the training cells' live state.

In [ ]:
# ============================================================
# Validation segment accuracy / frame accuracy / boundary F1 / loss
# ============================================================
curves = {}
for name in CONFIG["runs"]:
    rd = run_dir_for(name)
    if rd and (rd / "assets" / "history.json").exists():
        curves[name] = json.loads((rd / "assets" / "history.json").read_text())
fig, axes = plt.subplots(1, 4, figsize=(18, 3.6))
for name, h in curves.items():
    for ax, key in zip(axes, ("val_acc", "val_frame_acc", "val_boundary_f1", "val_loss")):
        ax.plot(np.arange(1, len(h[key]) + 1), h[key], label=name)
for ax, title in zip(axes, ("val segment accuracy (selection)", "val frame accuracy", "val boundary F1 (±3)", "val loss")):
    ax.set_title(title)
    ax.set_xlabel("epoch")
axes[0].legend()
fig.tight_layout()
fig.savefig(ASSETS / "learning_curves.png", dpi=110)
plt.show()
summary = pd.DataFrame({n: {"epochs": len(h["val_acc"]), "best_seg_acc": max(h["val_acc"]),
                            "best_epoch": int(np.argmax(h["val_acc"])) + 1,
                            "frame_acc@best": h["val_frame_acc"][int(np.argmax(h["val_acc"]))],
                            "boundary_f1@best": h["val_boundary_f1"][int(np.argmax(h["val_acc"]))]}
                        for n, h in curves.items()}).T
print(summary.to_string())

## 8. Canonical isolated evaluation

The same `sb-evaluate` every registry run gets: GISLR_Stratified `test.csv`,
per-class accuracy, the model read out at each clip's last frame. It puts
the continuous models on the existing leaderboard next to the isolated
ones (`gru_reg` 0.7517). Note that `sb-evaluate` subsamples clips longer
than 128 frames, as it does for every model. Copen scores 0 on its 20
held-out glosses by construction.

In [ ]:
# ============================================================
# sb-evaluate every finished run (skips runs already canonical)
# ============================================================
from sb.mlops import registry as R
from sb.recognize.evaluate import evaluate_run

for name in CONFIG["runs"]:
    rd = run_dir_for(name)
    if rd is None or not (rd / "meta.json").exists():
        print(f"{name}: not trained yet")
        continue
    meta = R.load_meta(rd)
    if not meta["training"]["finished"]:
        print(f"{name}: still training ({meta['training']['epochs_trained']} epochs)")
        continue
    if meta["metrics"]["eval_status"] != "canonical":
        evaluate_run(rd, verbose=False)
        meta = R.load_meta(rd)
    m = meta["metrics"]
    print(f"{name:<6} run {rd.name}: canonical accuracy {m['overall_accuracy']:.4f} · macro {m['macro_accuracy']:.4f}")